In [ ]:
import pandas as pd
import os

INPUT  = r"C:\Users\asus\Desktop\New_Dataset\AQI_Birendranagar_Surkhet.csv"
OUTPUT = r"C:\Users\asus\Desktop\New_Dataset\AQI_Birendranagar_Surkhet_Cleaned.csv"

df = pd.read_csv(INPUT)

print("="*55)
print("  CLEANING: Birendranagar Surkhet AQI")
print("="*55)
print(f"  Raw rows     : {len(df):,}")
print(f"  Parameters   : {df['parameter'].unique().tolist()}")

df["datetime_utc"]   = pd.to_datetime(df["datetime_utc"],   utc=True,   errors="coerce")
df["datetime_local"] = pd.to_datetime(df["datetime_local"],              errors="coerce")

df = df.dropna(subset=["value", "datetime_utc"])
print(f"  After null drop  : {len(df):,} rows")

before = len(df)
df = df[~((df["parameter"] == "pm1")              & (df["value"] < 0))]
df = df[~((df["parameter"] == "pm1")              & (df["value"] > 500))]
df = df[~((df["parameter"] == "pm25")             & (df["value"] < 0))]
df = df[~((df["parameter"] == "pm25")             & (df["value"] > 999))]
df = df[~((df["parameter"] == "temperature")      & (df["value"] < -10))]
df = df[~((df["parameter"] == "temperature")      & (df["value"] > 60))]
df = df[~((df["parameter"] == "relativehumidity") & (df["value"] < 0))]
df = df[~((df["parameter"] == "relativehumidity") & (df["value"] > 100))]
df = df[~((df["parameter"] == "um003")            & (df["value"] < 0))]
print(f"  After outliers   : {len(df):,} rows ({before - len(df)} removed)")

pm1_sensors = df[df["parameter"] == "pm1"][["datetime_utc", "sensor_id"]].rename(
    columns={"sensor_id": "sensor_id_pm1"}
)

df_wide = df.pivot_table(
    index   = ["datetime_utc", "datetime_local"],
    columns = "parameter",
    values  = "value",
    aggfunc = "mean"
).reset_index()

df_wide.columns.name = None
print(f"  After pivot      : {len(df_wide):,} rows")

df_wide = df_wide.merge(pm1_sensors, on="datetime_utc", how="left")

df_wide["city"]          = "Birendranagar"
df_wide["latitude"]      = 28.6000
df_wide["longitude"]     = 81.6167
df_wide["location_name"] = "Birendranagar_Surkhet_Ward12"

df_wide["date"]       = df_wide["datetime_utc"].dt.strftime("%Y-%m-%d")
df_wide["year"]       = df_wide["datetime_utc"].dt.year
df_wide["month"]      = df_wide["datetime_utc"].dt.month
df_wide["year_month"] = df_wide["datetime_utc"].dt.to_period("M").astype(str)

param_cols = ["pm1", "pm25", "temperature", "relativehumidity", "um003"]
param_cols = [c for c in param_cols if c in df_wide.columns]
df_wide    = df_wide.dropna(subset=param_cols, how="all")

df_wide = df_wide.rename(columns={
    "pm1"              : "Parameter(pm1)",
    "pm25"             : "Parameter(pm2.5)",
    "temperature"      : "temperature",
    "relativehumidity" : "relativehumidity",
    "um003"            : "um003",
})

final_cols = [
    "sensor_id_pm1",
    "datetime_utc",
    "datetime_local",
    "Parameter(pm1)",
    "Parameter(pm2.5)",
    "temperature",
    "relativehumidity",
    "um003",
    "city",
    "latitude",
    "longitude",
    "date",
    "year",
    "month",
    "year_month",
]

final_cols = [c for c in final_cols if c in df_wide.columns]
df_wide    = df_wide[final_cols]
df_wide    = df_wide.rename(columns={"sensor_id_pm1": "sensor_id"})
df_wide    = df_wide.sort_values("datetime_utc").reset_index(drop=True)

df_wide.to_csv(OUTPUT, index=False)

print(f"\n  SAVED    : {OUTPUT}")
print(f"  Rows     : {len(df_wide):,}")
print(f"  Columns  : {df_wide.columns.tolist()}")
print(f"\n  Stats:")
for col in ["Parameter(pm1)", "Parameter(pm2.5)", "temperature", "relativehumidity", "um003"]:
    if col in df_wide.columns:
        print(f"  {col:25} mean={df_wide[col].mean():.2f}  min={df_wide[col].min():.2f}  max={df_wide[col].max():.2f}  nulls={df_wide[col].isnull().sum()}")
print(f"\n  Rows per month:")
print(df_wide.groupby("year_month").size().to_string())

Saved : C:\Users\asus\Desktop\New_Dataset\COMBINED_SURKHET.csv
Rows  : 5,288
Cols  : ['sensor_id', 'datetime_utc', 'datetime_local', 'Parameter(pm1)', 'Parameter(pm2.5)', 'temperature', 'relativehumidity', 'rainfall (mm)', 'humidity (%)', 'windspeed (km/h)', 'city', 'latitude', 'longitude', 'date']
From  : 2025-10-26 to 2026-06-30
    sensor_id               datetime_utc             datetime_local  Parameter(pm1)  Parameter(pm2.5)  temperature  relativehumidity  rainfall (mm)  humidity (%)  windspeed (km/h)           city  latitude  longitude        date
0  14476518.0  2025-10-26 09:00:00+00:00  2025-10-26 14:45:00+05:45       29.281750         50.844084    29.382050         41.755000            0.0            51               7.3  Birendranagar      28.6    81.6167  2025-10-26
1  14476518.0  2025-10-26 10:00:00+00:00  2025-10-26 15:45:00+05:45       28.375319         48.325486    32.657556         35.633944            0.0            51               7.3  Birendranagar      28.6    81.